In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import BertTokenizer, BertModel
import torchvision.models as models
from sklearn.metrics import accuracy_score, classification_report
import shap
import os

!pip install transformers shap gradio -q
print("✅ Ready")

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
id2label  = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Fusion model ──
class MultimodalFusionModel(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        resnet = models.resnet50(pretrained=True)
        self.image_encoder = nn.Sequential(*list(resnet.children())[:-1])
        self.fusion = nn.Sequential(
            nn.Linear(2816, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 128),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_ids, attention_mask, images):
        text_feat = self.bert(input_ids=input_ids,
                              attention_mask=attention_mask).pooler_output
        img_feat  = self.image_encoder(images).view(images.size(0), -1)
        return self.fusion(torch.cat([text_feat, img_feat], dim=1))

fusion_model = MultimodalFusionModel()
fusion_model.load_state_dict(
    torch.load('/content/drive/MyDrive/MultimodalSentiment/models/best_fusion.pth',
               map_location=device))
fusion_model = fusion_model.to(device)
fusion_model.eval()
print("✅ Fusion model loaded")

In [ ]:
# Simple predict function for SHAP
def predict_text(texts):
    encodings = tokenizer(
        list(texts), max_length=128, padding='max_length',
        truncation=True, return_tensors='pt')
    input_ids      = encodings['input_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)
    images         = torch.zeros(len(texts), 3, 224, 224).to(device)

    with torch.no_grad():
        outputs = fusion_model(input_ids, attention_mask, images)
        probs   = torch.softmax(outputs, dim=1).cpu().numpy()
    return probs

# Sample sentences
samples = [
    "I absolutely love this show, it makes me so happy!",
    "This is the worst day of my life.",
    "I don't know how I feel about this.",
    "That was surprisingly amazing!",
    "I can't believe they did that to me."
]

# SHAP explainer
masker   = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(predict_text, masker,
                           output_names=['Negative','Neutral','Positive'])
shap_values = explainer(samples)

# Plot
shap.plots.text(shap_values[0])  # first sample
plt.savefig('/content/drive/MyDrive/MultimodalSentiment/outputs/shap_chart.png',
            dpi=150, bbox_inches='tight')
print("✅ SHAP chart saved!")

In [ ]:
BASE = '/content/drive/MyDrive/MultimodalSentiment/data/'
test_df = pd.read_csv(BASE + 'test_sent_emo.csv')
test_df = test_df[test_df['Sentiment'].isin(label_map.keys())].reset_index(drop=True)

# Get predictions on test samples
all_preds, all_true, all_texts = [], [], []

fusion_model.eval()
with torch.no_grad():
    for idx in range(len(test_df)):
        row  = test_df.iloc[idx]
        text = str(row['Utterance'])
        enc  = tokenizer(text, max_length=128, padding='max_length',
                         truncation=True, return_tensors='pt')
        iids = enc['input_ids'].to(device)
        amsk = enc['attention_mask'].to(device)
        imgs = torch.zeros(1, 3, 224, 224).to(device)
        out  = fusion_model(iids, amsk, imgs)
        pred = out.argmax(1).item()
        true = label_map[row['Sentiment']]
        all_preds.append(pred)
        all_true.append(true)
        all_texts.append(text)

# Find wrong predictions
errors = [(all_texts[i], id2label[all_true[i]], id2label[all_preds[i]])
          for i in range(len(all_preds)) if all_preds[i] != all_true[i]]

print(f"Total errors: {len(errors)} / {len(test_df)}")
print(f"Test Accuracy: {accuracy_score(all_true, all_preds):.4f}")
print("\n=== Sample Wrong Predictions ===\n")
for text, true, pred in errors[:8]:
    print(f"Text:      {text}")
    print(f"True:      {true}")
    print(f"Predicted: {pred}")
    print("-"*60)

In [ ]:
import gradio as gr
from PIL import Image as PILImage
import torchvision.transforms as transforms

img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def predict_sentiment(text, image=None):
    # Text
    enc = tokenizer(text, max_length=128, padding='max_length',
                    truncation=True, return_tensors='pt')
    input_ids      = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)

    # Image
    if image is not None:
        img_tensor = img_transform(
            PILImage.fromarray(image).convert('RGB')).unsqueeze(0).to(device)
    else:
        img_tensor = torch.zeros(1, 3, 224, 224).to(device)

    fusion_model.eval()
    with torch.no_grad():
        out   = fusion_model(input_ids, attention_mask, img_tensor)
        probs = torch.softmax(out, dim=1).cpu().numpy()[0]

    return {
        'Negative': float(probs[0]),
        'Neutral':  float(probs[1]),
        'Positive': float(probs[2])
    }

demo = gr.Interface(
    fn=predict_sentiment,
    inputs=[
        gr.Textbox(label="Enter utterance", placeholder="Type something..."),
        gr.Image(label="Upload face image (optional)")
    ],
    outputs=gr.Label(num_top_classes=3, label="Sentiment"),
    title="🎭 Multimodal Sentiment Analyzer",
    description="Combines BERT (text) + ResNet-50 (image) to predict sentiment",
    examples=[
        ["I absolutely love this, it makes me so happy!", None],
        ["This is terrible, I hate everything about it.", None],
        ["I don't really know how to feel about this.", None]
    ]
)

demo.launch(share=True)  # share=True gives public URL for viva demo